# Predicting a bike's category from its specs

17 classes (Scooter, Sport, Enduro / offroad, Custom / cruiser, ...) predicted from engine,
weight and geometry specs. The labels already exist in the dataset, so we can score the model
against the truth.

Data comes from `data/processed/bikes.parquet`. Build it first if it is missing:

```
uv run python -m two_wheel_data.util.clean
```

The one trap worth knowing: the same model appears in several model years with near-identical
specs, so a random train/test split leaks twins across the split and inflates the score. We split
by `Model` instead.

In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit

from two_wheel_data.data import load_processed

df = load_processed()
print(df.shape)
df["Category"].value_counts()

## Features

Every numeric spec the cleaning step parsed, plus the low-cardinality text columns. `Rating` is
left out: it is a crowd score, not a spec, and it is missing for 43% of bikes.

The high-cardinality text columns (`Fuel system`, `Frame type`, `Front brakes`, ... up to 4,652
distinct values) are skipped for now - they need mining into flags like `has_abs` first.

`HistGradientBoostingClassifier` handles NaN natively, so rows with missing specs stay in.

In [ ]:
MAX_CATEGORIES = 255  # HistGradientBoosting's limit per categorical feature

labelled = df[df["Category"] != "Unspecified category"].dropna(subset=["Model"])

numeric_cols = [c for c in labelled.select_dtypes("number").columns if c != "Rating"]
categorical_cols = [
    c
    for c in ["Engine type", "Cooling system", "Gearbox", "Transmission type", "Fuel control", "Starter"]
    if labelled[c].nunique() <= MAX_CATEGORIES
]

X = labelled[numeric_cols + categorical_cols].copy()
for col in categorical_cols:
    X[col] = X[col].astype("category")

y = labelled["Category"]
groups = labelled["Model"]

print(f"{len(numeric_cols)} numeric + {len(categorical_cols)} categorical features")
X.head()

## Grouped split

`GroupShuffleSplit` keeps every year of a given model on one side of the split. The overlap check
below should print 0 - with a plain `train_test_split` it would be in the thousands.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

shared_models = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"train={len(train_idx)}  test={len(test_idx)}  models in both: {len(shared_models)}")

## Baseline first

Always guessing the most common class (Scooter) is the number to beat.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print(f"accuracy  {accuracy_score(y_test, baseline_pred):.3f}")
print(f"macro F1  {f1_score(y_test, baseline_pred, average='macro'):.3f}")

## Gradient boosting

Macro F1 averages per-class scores equally, so the rare classes count as much as Scooter. It is the
honest metric here; accuracy alone flatters a model on an imbalanced dataset.

In [ ]:
model = HistGradientBoostingClassifier(
    categorical_features="from_dtype", random_state=0
).fit(X_train, y_train)
pred = model.predict(X_test)

print(f"accuracy  {accuracy_score(y_test, pred):.3f}")
print(f"macro F1  {f1_score(y_test, pred, average='macro'):.3f}")
print(classification_report(y_test, pred, zero_division=0))

## Where it goes wrong

The biggest confusions are pairs that overlap in reality too - a Naked bike and a Sport bike can
share a platform, and Allround is a catch-all.

In [ ]:
confusion = pd.crosstab(y_test, pred, rownames=["actual"], colnames=["predicted"])
mistakes = [
    (actual, predicted, confusion.loc[actual, predicted])
    for actual in confusion.index
    for predicted in confusion.columns
    if actual != predicted and confusion.loc[actual, predicted] > 0
]
pd.DataFrame(
    sorted(mistakes, key=lambda row: -row[2])[:10],
    columns=["actual", "predicted", "count"],
)

## Things to try next

- Clip implausible values first (`Seat height (mm)` maxes at 75,010; `Valves per cylinder` at
  5,599) - they are data-entry errors the model currently treats as real.
- Compare against `LogisticRegression` in a `Pipeline` (needs imputing + scaling + one-hot) to see
  what the boosting model buys you.
- `permutation_importance` to find which specs carry the signal.
- `class_weight="balanced"` or merging tiny classes (Speedway has ~20 rows) to lift macro F1.
- Swap `GroupShuffleSplit` for `GroupKFold` + `cross_val_score` for a more stable estimate.
- Mine the free-text columns (`Front brakes`, `Fuel system`) into flags like `has_abs`.